In [0]:

from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df = spark.read.format("delta")\
    .load("/Volumes/sparkcatalog/raw/source/delta/")
df.show()

In [0]:
df.createTempView("names")
df_sql = spark.sql("""SELECT * FROM names""")
display(df_sql)

In [0]:
df_sql = spark.sql("""UPDATE names SET Greet=Null WHERE  id=2""")
display(df_sql)

In [0]:
df = spark.read.format("delta")\
    .load("/Volumes/sparkcatalog/raw/source/delta/")
df.show()

In [0]:
df_fix = spark.sql("""SELECT * FROM names WHERE Greet IS NULL""")
display(df_fix)

In [0]:
df_f = df_sql.fillna({"Greet":"Manikandan"})
df_f.show()


In [0]:
df.write.format("delta")\
    .mode("append")\
    .save("/Volumes/sparkcatalog/raw/source/delta/")
df = spark.read.format("delta")\
    .load("/Volumes/sparkcatalog/raw/source/delta/")
df.show()

In [0]:
spark.sql("""CREATE TABLE IF NOT EXISTS sparkcatalog.raw.table_1(
    id INT PRIMARY KEY,
    name VARCHAR(50),
    age INT 
)""")

spark.sql("""INSERT INTO sparkcatalog.raw.table_1
          VALUES
          (1,'Manikandan',20),
          (2,'Priya',19)
          """)

#UPSERT USING SPARK SQL

In [0]:
display(spark.sql("""SELECT * FROM sparkcatalog.raw.table_1"""))

In [0]:
data = [(1,"jhon",30),
        (2,"peter",50),
        (3,"jack",20),
        (4,"mary",40),
        (5,"jane",30),]
df_2 = spark.createDataFrame(data,"id INT, name STRING, age INT")
display(df_2)

In [0]:
df_2.createOrReplaceTempView("Boobie")
spark.sql("""MERGE INTO sparkcatalog.raw.table_1 t
                USING Boobie s
                ON t.id = s.id
                WHEN MATCHED THEN 
                    UPDATE SET * 
                WHEN NOT MATCHED THEN
                    INSERT *
          """)
display(spark.sql("""SELECT * FROM sparkcatalog.raw.table_1"""))

#Merge Into Api UPSERT

In [0]:
data1 = [(7,"jhoneSince",30),
        (8,"peter parker",25),
        (3,"jack",20),
        (4,"mary",40),
        (5,"jane",30),]
df_3 = spark.createDataFrame(data1,"id INT, name STRING, age INT")
display(df_3)

In [0]:
df_3.alias("src").mergeInto("sparkcatalog.raw.table_1",col("src.id")==col("sparkcatalog.raw.table_1.id"))\
    .whenMatched().updateAll()\
    .whenNotMatched().insertAll()\
    .merge()

In [0]:

display(spark.sql("""SELECT * FROM sparkcatalog.raw.table_1
                  ORDER BY id ASC"""))

#PARTITION 

In [0]:
spark.sql("""CREATE TABLE IF NOT EXISTS sparkcatalog.raw.table_2(
    id INT PRIMARY KEY,
    product VARCHAR(50),
    price INT
)
USING DELTA
PARTITIONED BY (id)
""")


In [0]:
%sql
INSERT INTO sparkcatalog.raw.table_2 VALUES 
    (
        1,
        'slatePencile',
        10
    ),
    (2,'Book',100),
    (3,'Pen',10)



In [0]:
%sql
SELECT * FROM sparkcatalog.raw.table_2

In [0]:
%sql
EXPLAIN SELECT * FROM sparkcatalog.raw.table_2

#SQL SCRIPT FUNCTIONS

In [0]:
%sql
DECLARE salary INT DEFAULT 500000;

SELECT salary;

In [0]:
%sql
BEGIN
DECLARE salarys INT DEFAULT 500000;
IF salarys >= 500000 THEN
    SELECT 'PARAVALLAYE';
ELSEIF salarys<50000 THEN
    SELECT 'OK';
ELSE 
    SELECT 'NOT OK';
END IF;
END;

In [0]:
%sql
SELECT * FROM sparkcatalog.raw.table_2

In [0]:
%sql
WITH products AS(
    SELECT *, 
    CASE
        WHEN price<100 THEN 'LOW PRICE'
        ELSE 'HIGH PRICE'
    END AS price_range
    FROM sparkcatalog.raw.table_2     
)
SELECT * FROM products

In [0]:
%sql

    SELECT *, 
    CASE
        WHEN price<100 THEN 'LOW PRICE'
        ELSE 'HIGH PRICE'
    END AS price_range
    FROM sparkcatalog.raw.table_2     


#Spark SQL Auxiliary Statements

In [0]:
%sql
DESCRIBE sparkcatalog.raw.table_2

In [0]:
%sql
SHOW DATABASES

In [0]:
%sql
SHOW TABLES

In [0]:
%sql
SHOW SCHEMAS

In [0]:
%sql
WITH sharWaste AS(
    SELECT id,
        product
        FROM sparkcatalog.raw.table_2
)
SELECT * FROM sharWaste WHERE id=1

In [0]:
%sql
USE sparkcatalog.raw

In [0]:
df = spark.read.format("csv")\
    .option("inforSchema",True)\
    .option("header",True)\
    .load("/Volumes/sparkcatalog/raw/source/raw_orders/orders.csv")

In [0]:
df.createOrReplaceTempView("sql_df")

In [0]:
%sql
SELECT * FROM sql_df

In [0]:
%sql
SELECT customer_id ,
    collect_list(order_id) AS order_list
FROM sql_df
GROUP BY customer_id

#STRUCK AND MAP

In [0]:
%sql
SELECT struct(1,2,3,4,"mani
")

In [0]:
%sql
SELECT
    NAMED_STRUCT(
        'city', 'Chennai',
        'age', 21
    ) AS details

In [0]:
%sql
SELECT
    MAP(
        'city', 'Chennai',
        'age', '21'
    ) AS details

#DATE TIMES

In [0]:
%sql
SELECT current_date(),
date_format(current_date(),"MM-dd-yy"),
date_format(date_add(current_date(),2),'dd-MM-yy')

#Arrayy Functions

In [0]:
%sql
SELECT array(1, 2, 3);

SELECT array_append(array(1,2,3), 4);

SELECT array_distinct(array(1,2,3,3,4));

SELECT array_max(array(1,2,3));

SELECT array_min(array(1,2,3));

SELECT array_position(array(1,2,3), 2);

SELECT array_insert(array(1,2,3), 2, 4);

SELECT array_remove(array(1,2,3), 2);

SELECT array_sort(array(1,2,3));

SELECT array_insert(array(1,2,3), 1, 100);

SELECT array_contains(array(111,222,33), 222);

SELECT array_prepend(array(10,12,15),9)
    

#UDF in Spark SQL

In [0]:
def Gaadi(input):
    if input == "car":
        return "15000 services"
    else:
        return "5000 services"

In [0]:
spark.udf.register("Gaadi",Gaadi)

In [0]:
data = [(1,"car","Shift"),
        (2,"Bike","Xpluse")]
df_car = spark.createDataFrame(data,["id","vehicle","model"])
df_car.createOrReplaceTempView("df_car")


In [0]:
%sql
SELECT *,
    gaadi(vehicle) AS service_charge
FROM df_car

#READ FILE IN SPARK SQL

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW sql_data
USING csv
OPTIONS (
    path "/Volumes/sparkcatalog/raw/source/raw_orders/rawData.csv",
    header "true",
    inferShema "true"
)

In [0]:
%sql
SELECT * FROM sql_data